# Exportação de Dados para Dashboard

Este notebook exporta métricas e resultados para o banco de dados, permitindo visualização no dashboard.


In [3]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from mlflow.tracking import MlflowClient
import os

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'postgres')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'diabetes_db')

DATABASE_URL = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(DATABASE_URL)

mlflow_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://mlflow:5000')
client = MlflowClient(mlflow_uri)

print("Conexões estabelecidas com sucesso!")


Conexões estabelecidas com sucesso!


## Exportar Estatísticas do Dataset


In [4]:
# Carregar dados
query = "SELECT * FROM diabetes_processed"
df = pd.read_sql(query, engine)

# Calcular estatísticas
stats = {
    'total': len(df),
    'n': len(df[df['class_label'] == 'N']),
    'p': len(df[df['class_label'] == 'P']),
    'y': len(df[df['class_label'] == 'Y']),
    'age': float(df['age'].mean()),
    'hba1c': float(df['hba1c'].mean()),
    'bmi': float(df['bmi'].mean())
}

# Limpar tabela e inserir novas estatísticas
with engine.connect() as conn:
    conn.execute(text("DELETE FROM dataset_stats"))
    conn.execute(text("""
        INSERT INTO dataset_stats 
        (total_records, class_n_count, class_p_count, class_y_count, 
         avg_age, avg_hba1c, avg_bmi)
        VALUES (:total, :n, :p, :y, :age, :hba1c, :bmi)
    """), stats)
    conn.commit()

print("✅ Estatísticas do dataset exportadas com sucesso!")
print(f"Total de registros: {stats['total']}")

✅ Estatísticas do dataset exportadas com sucesso!
Total de registros: 3000


## Exportar Métricas dos Modelos do MLFlow


In [5]:
# Buscar experimentos do MLFlow
experiments = client.search_experiments()

all_metrics = []

for exp in experiments:
    if 'diabetes' in exp.name.lower():
        runs = client.search_runs(experiment_ids=[exp.experiment_id], max_results=20)
        
        for run in runs:
            metrics = run.data.metrics
            params = run.data.params
            
            # Extrair métricas
            model_metrics = {
                'model_name': params.get('model', run.info.run_name),
                'accuracy': float(metrics.get('accuracy', 0.0)),
                'f1_score_weighted': float(metrics.get('f1_score_weighted', 0.0)),
                'f1_score_macro': float(metrics.get('f1_score_macro', 0.0)),
                'precision_n': float(metrics.get('precision_N', 0.0)),
                'precision_p': float(metrics.get('precision_P', 0.0)),
                'precision_y': float(metrics.get('precision_Y', 0.0)),
                'recall_n': float(metrics.get('recall_N', 0.0)),
                'recall_p': float(metrics.get('recall_P', 0.0)),
                'recall_y': float(metrics.get('recall_Y', 0.0)),
                'mlflow_run_id': run.info.run_id
            }
            
            all_metrics.append(model_metrics)

if all_metrics:
    metrics_df = pd.DataFrame(all_metrics)
    
    # Limpar e inserir no banco
    with engine.connect() as conn:
        conn.execute(text("DELETE FROM model_metrics"))
    metrics_df.to_sql('model_metrics', engine, if_exists='append', index=False)
    
    print(f"✅ {len(metrics_df)} modelos exportados com sucesso!")
    print("\nMétricas exportadas:")
    print(metrics_df[['model_name', 'accuracy', 'f1_score_weighted']].to_string(index=False))
else:
    print("⚠️ Nenhuma métrica encontrada no MLFlow")


✅ 23 modelos exportados com sucesso!

Métricas exportadas:
                   model_name  accuracy  f1_score_weighted
Gradient Boosting - Optimized  1.000000           1.000000
    Random Forest - Optimized  1.000000           1.000000
    Random Forest - Optimized  0.995000           0.994876
                          KNN  0.975000           0.975791
          Logistic Regression  0.926667           0.933218
                          SVM  0.981667           0.982093
            Gradient Boosting  1.000000           1.000000
                Random Forest  1.000000           1.000000
                          KNN  0.975000           0.975791
          Logistic Regression  0.926667           0.933218
                          SVM  0.981667           0.982093
            Gradient Boosting  1.000000           1.000000
                Random Forest  1.000000           1.000000
                          KNN  0.940000           0.945358
          Logistic Regression  0.935000           0.9385

## Verificação Final


In [6]:
# Verificar dados exportados
print("=" * 60)
print("VERIFICAÇÃO DOS DADOS EXPORTADOS")
print("=" * 60)

# Estatísticas
stats_query = "SELECT * FROM dataset_stats"
stats_df = pd.read_sql(stats_query, engine)
print("\n📊 Estatísticas do Dataset:")
print(stats_df.to_string(index=False))

# Métricas
metrics_query = "SELECT model_name, accuracy, f1_score_weighted FROM model_metrics ORDER BY accuracy DESC"
metrics_df = pd.read_sql(metrics_query, engine)
print("\n🤖 Métricas dos Modelos:")
if not metrics_df.empty:
    print(metrics_df.to_string(index=False))
else:
    print("Nenhuma métrica encontrada")

# Predições
pred_query = "SELECT COUNT(*) as total FROM model_predictions"
pred_count = pd.read_sql(pred_query, engine)
print(f"\n📉 Total de Predições: {pred_count['total'].iloc[0]}")

print("\n✅ Dados exportados com sucesso!")
print("\nAcesse o dashboard em: http://localhost:8501")


VERIFICAÇÃO DOS DADOS EXPORTADOS

📊 Estatísticas do Dataset:
 id  total_records  class_n_count  class_p_count  class_y_count  avg_age  avg_hba1c  avg_bmi                 updated_at
  1           3000            309            159           2532   53.528    8.28116 29.57802 2025-11-24 18:59:06.215821

🤖 Métricas dos Modelos:
                   model_name  accuracy  f1_score_weighted
Gradient Boosting - Optimized  1.000000           1.000000
    Random Forest - Optimized  1.000000           1.000000
            Gradient Boosting  1.000000           1.000000
                Random Forest  1.000000           1.000000
            Gradient Boosting  1.000000           1.000000
                Random Forest  1.000000           1.000000
    Random Forest - Optimized  0.995000           0.994876
            Gradient Boosting  0.995000           0.994876
                Random Forest  0.995000           0.994876
            Gradient Boosting  0.995000           0.994876
                Random Fo